# Exercise P3.2: Web Scraping with BeautifulSoup
### STAT 540 — Week 3


## Overview

In this exercise, you will scrape structured data from websites using BeautifulSoup, clean the results, and combine data from multiple pages.

## Task 1: Scrape books.toscrape.com

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

response = requests.get("https://books.toscrape.com")
soup = BeautifulSoup(response.content, "html.parser")

books = []
for article in soup.select("article.product_pod"):
    title = article.select_one("h3 a")["title"]
    price = article.select_one(".price_color").text
    rating = article.select_one("p.star-rating")["class"][1]
    books.append({"title": title, "price": price, "rating": rating})

df = pd.DataFrame(books)
print(df.head(10))

                                               title   price rating
0                               A Light in the Attic  £51.77  Three
1                                 Tipping the Velvet  £53.74    One
2                                         Soumission  £50.10    One
3                                      Sharp Objects  £47.82   Four
4              Sapiens: A Brief History of Humankind  £54.23   Five
5                                    The Requiem Red  £22.65    One
6  The Dirty Little Secrets of Getting Your Dream...  £33.34   Four
7  The Coming Woman: A Novel Based on the Life of...  £17.93  Three
8  The Boys in the Boat: Nine Americans and Their...  £22.60   Four
9                                    The Black Maria  £52.15    One


**Your turn:** How many books are on the first page? What is the most common star rating?

> There are 10 books on the first page, with the most common rating being One.

## Task 2: Clean the Price Column

In [2]:
# Convert price from "£51.77" to numeric
df["price_numeric"] = df["price"].str.replace("£", "", regex=False).astype(float)

print(f"Price range: £{df['price_numeric'].min():.2f} to £{df['price_numeric'].max():.2f}")
print(f"Mean price: £{df['price_numeric'].mean():.2f}")

Price range: £13.99 to £57.25
Mean price: £38.05


## Task 3: Scrape Multiple Pages

In [3]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time

all_books = []
for page_num in range(1, 4):
    url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
    response = requests.get(url)

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.content, "html.parser")
    for article in soup.select("article.product_pod"):
        title = article.select_one("h3 a")["title"]
        price = article.select_one(".price_color").text.replace("£", "")
        rating = article.select_one("p.star-rating")["class"][1]
        all_books.append({"title": title, "price": float(price), "rating": rating})

    print(f"Page {page_num}: {len(all_books)} books total")
    time.sleep(1)

df_all = pd.DataFrame(all_books)
print(f"\nTotal books: {len(df_all)}")
print(f"\nRating distribution:\n{df_all['rating'].value_counts()}")

Page 1: 20 books total
Page 2: 40 books total
Page 3: 60 books total

Total books: 60

Rating distribution:
rating
One      15
Five     14
Three    13
Four     10
Two       8
Name: count, dtype: int64


## Task 4: Scrape a Wikipedia Table

In [5]:
import pandas as pd
import requests

# pandas.read_html is the fastest way to get HTML tables
url = "https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_population"

# Add User-Agent header to mimic a browser and avoid 403 Forbidden error
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
response = requests.get(url, headers=headers)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

tables = pd.read_html(response.text)
print(f"Found {len(tables)} tables")

states = tables[0]
print(states.head())

Found 6 tables
  Unnamed: 0_level_0 State or territory Census population[8][9][a]  \
  Unnamed: 0_level_1 State or territory        July 1, 2025 (est.)   
0                  1         California                 39355309.0   
1                  2              Texas                 31709821.0   
2                  3            Florida                 23462518.0   
3                  4           New York                 20002427.0   
4                  5       Pennsylvania                 13059432.0   

                House seats[b]         Pop. per elec. vote (2020)[c]  \
  April 1, 2020          Seats       % Pop. per elec. vote (2020)[c]   
0      39538223             52  11.95%                        732189   
1      29145505             38   8.74%                        728638   
2      21538187             28   6.44%                        717940   
3      20201249             26   5.98%                        721473   
4      13002700             17   3.91%                        

/tmp/ipykernel_4010/179385007.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


**Your turn:** Compare the effort of scraping this table with `pd.read_html()` versus using BeautifulSoup. When would you need BeautifulSoup instead?

> Using `read_html` is much easier and takes just a single lign of code. However, BeautifulSoup is necessary when you need to go beyond basic text extraction from standard table tags.

## Task 5: Compare R and Python Scraping

Based on R3.2 and this exercise, fill in:

| Task | R (rvest) | Python (BeautifulSoup) |
|------|-----------|----------------------|
| Load page | `read_html()` | `requests.get()` |
| Select elements | `html_nodes()` / `html_elements()` | `soup.select()` / `soup.find_all()` |
| Get text | `html_text()` | `.text` |
| Get attribute | `html_attr()` | `['attribute_name']` |
| Parse tables | `html_table()` | `pd.read_html()` |


```bash
git add week03/exercises/P3.2*
git commit -m "Complete Exercise P3.2: BeautifulSoup web scraping"
git push origin main
```